In [ ]:
# Análise Comparativa: STI vs MTI - Métricas de Preferência Humana

Este notebook apresenta uma análise visual detalhada das métricas comparativas entre as abordagens **Single-Task Inference (STI)** e **Multi-Task Inference (MTI)** para diferentes modelos de linguagem.

## Métricas Avaliadas:
- **Coherence** (Coerência)
- **Specificity** (Especificidade)
- **Informativeness** (Informatividade)
- **Relevance** (Relevância)
- **Understandability** (Compreensibilidade)

Todas as métricas são avaliadas em uma escala Likert de 1 a 5.

## 1. Importação de Bibliotecas e Conexão com MongoDB

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pymongo import MongoClient
from dotenv import load_dotenv

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Carregar variáveis de ambiente
load_dotenv()

# Conexão MongoDB
MONGO_URI = os.getenv("MONGODB_URI")
mongo_client = MongoClient(MONGO_URI)
db = mongo_client["experiments_db"]
comparative_metrics_collection = db["llm_judge_comparative_metrics"]

# Métricas avaliadas
METRICS = ["coherence", "specificity", "informativeness", "relevance", "Understandability"]
METRIC_LABELS = {
    "coherence": "Coherence\n(Coerência)",
    "specificity": "Specificity\n(Especificidade)",
    "informativeness": "Informativeness\n(Informatividade)",
    "relevance": "Relevance\n(Relevância)",
    "Understandability": "Understandability\n(Compreensibilidade)"
}

print("✅ Bibliotecas importadas e conexão MongoDB estabelecida!")

✅ Bibliotecas importadas e conexão MongoDB estabelecida!


## 2. Carregamento e Processamento dos Dados

In [2]:
# Carregar todos os documentos da collection
documents = list(comparative_metrics_collection.find())

print(f"📊 Total de experimentos encontrados: {len(documents)}\n")

# Criar DataFrames para análise
data_rows = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    
    # Extrair nome do modelo (removendo prefixo "LLM_Judge_")
    model_name = experiment_name.replace("LLM_Judge_", "")
    
    # Para cada métrica individual
    for metric in METRICS:
        # Dados MTI
        mti_stats = doc["MTI_metrics"][metric]
        data_rows.append({
            "experiment": experiment_name,
            "model": model_name,
            "metric": metric,
            "approach": "MTI",
            "mean": mti_stats["mean"],
            "std": mti_stats["std"],
            "count_max_score": mti_stats["count_max_score"],
            "total_samples": mti_stats["total_samples"]
        })
        
        # Dados STI
        sti_stats = doc["STI_metrics"][metric]
        data_rows.append({
            "experiment": experiment_name,
            "model": model_name,
            "metric": metric,
            "approach": "STI",
            "mean": sti_stats["mean"],
            "std": sti_stats["std"],
            "count_max_score": sti_stats["count_max_score"],
            "total_samples": sti_stats["total_samples"]
        })
    
    # Dados gerais (média de todas as métricas)
    overall_mti = doc["overall_statistics"]["MTI"]
    data_rows.append({
        "experiment": experiment_name,
        "model": model_name,
        "metric": "Overall (Média Geral)",
        "approach": "MTI",
        "mean": overall_mti["mean"],
        "std": overall_mti["std"],
        "count_max_score": overall_mti["count_max_score"],
        "total_samples": overall_mti["total_samples"]
    })
    
    overall_sti = doc["overall_statistics"]["STI"]
    data_rows.append({
        "experiment": experiment_name,
        "model": model_name,
        "metric": "Overall (Média Geral)",
        "approach": "STI",
        "mean": overall_sti["mean"],
        "std": overall_sti["std"],
        "count_max_score": overall_sti["count_max_score"],
        "total_samples": overall_sti["total_samples"]
    })

# Criar DataFrame
df = pd.DataFrame(data_rows)

print("✅ Dados processados com sucesso!")
print(f"\n📋 Experimentos (modelos) encontrados:")
for model in df["model"].unique():
    print(f"   • {model}")

df.head(100)

📊 Total de experimentos encontrados: 5

✅ Dados processados com sucesso!

📋 Experimentos (modelos) encontrados:
   • gpt-4o-mini_TEST
   • gpt-4o-mini
   • llama-3.3-70b-versatile
   • gpt-3.5-turbo-0125
   • llama2:7b


,experiment,model,metric,approach,mean,std,count_max_score,total_samples
0,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,coherence,MTI,4.40,0.55,2,5
1,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,coherence,STI,4.60,0.55,3,5
2,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,specificity,MTI,3.60,0.55,0,5
3,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,specificity,STI,4.00,0.00,0,5
4,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,informativeness,MTI,4.00,0.00,0,5
5,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,informativeness,STI,4.20,0.45,1,5
6,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,relevance,MTI,5.00,0.00,5,5
7,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,relevance,STI,5.00,0.00,5,5
8,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,Understandability,MTI,4.40,0.55,2,5
9,LLM_Judge_gpt-4o-mini_TEST,gpt-4o-mini_TEST,Understandability,STI,4.60,0.55,3,5


## 3. Visualizações - Média Geral (Overall) por Modelo e Abordagem

Comparação das médias gerais (todas as métricas combinadas) entre MTI e STI para cada modelo.

In [3]:
# Filtrar dados da média geral
df_overall = df[df["metric"] == "Overall (Média Geral)"].copy()

# Ordenar por abordagem (MTI primeiro, depois STI) e depois por média
df_overall_sorted = df_overall.sort_values(["approach", "mean"], ascending=[False, False])

# Gráfico de barras agrupadas - Média Geral
fig = go.Figure()

# Adicionar barras MTI (primeiro)
df_mti = df_overall_sorted[df_overall_sorted["approach"] == "MTI"]
fig.add_trace(go.Bar(
    name="MTI",
    x=df_mti["model"],
    y=df_mti["mean"],
    error_y=dict(type='data', array=df_mti["std"]),
    marker_color='rgb(55, 83, 109)',
    text=[f'{val:.3f}' for val in df_mti["mean"]],
    textposition='outside',
))

# Adicionar barras STI (depois)
df_sti = df_overall_sorted[df_overall_sorted["approach"] == "STI"]
fig.add_trace(go.Bar(
    name="STI",
    x=df_sti["model"],
    y=df_sti["mean"],
    error_y=dict(type='data', array=df_sti["std"]),
    marker_color='rgb(26, 118, 255)',
    text=[f'{val:.3f}' for val in df_sti["mean"]],
    textposition='outside',
))

fig.update_layout(
    title="Média Geral das Métricas: MTI vs STI por Modelo",
    xaxis_title="Modelo de Linguagem",
    yaxis_title="Pontuação Média (escala 1-5)",
    barmode='group',
    height=500,
    yaxis=dict(range=[0, 5.5]),
    legend=dict(x=0.01, y=0.99),
    template="plotly_white"
)

fig.show()

In [4]:
# Gráfico de pontos com barras de erro - Média Geral
fig = go.Figure()

for model in df_overall["model"].unique():
    df_model = df_overall[df_overall["model"] == model]
    
    # MTI primeiro
    df_mti_model = df_model[df_model["approach"] == "MTI"]
    if not df_mti_model.empty:
        fig.add_trace(go.Scatter(
            x=["MTI"],
            y=df_mti_model["mean"],
            error_y=dict(type='data', array=df_mti_model["std"]),
            mode='markers',
            marker=dict(size=12),
            name=f"{model} (MTI)",
            legendgroup=model,
        ))
    
    # STI depois
    df_sti_model = df_model[df_model["approach"] == "STI"]
    if not df_sti_model.empty:
        fig.add_trace(go.Scatter(
            x=["STI"],
            y=df_sti_model["mean"],
            error_y=dict(type='data', array=df_sti_model["std"]),
            mode='markers',
            marker=dict(size=12),
            name=f"{model} (STI)",
            legendgroup=model,
        ))

fig.update_layout(
    title="Distribuição das Médias Gerais: MTI vs STI",
    xaxis_title="Abordagem",
    yaxis_title="Pontuação Média (escala 1-5)",
    height=500,
    yaxis=dict(range=[0, 5.5]),
    template="plotly_white"
)

fig.show()

## 4. Visualizações - Métricas Individuais por Modelo

In [5]:
# Filtrar dados das métricas individuais
df_metrics = df[df["metric"] != "Overall (Média Geral)"].copy()

# Criar subplots para cada métrica
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[METRIC_LABELS[m] for m in METRICS] + [""],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

row_col_map = [(1,1), (1,2), (2,1), (2,2), (3,1)]

for idx, metric in enumerate(METRICS):
    row, col = row_col_map[idx]
    
    df_metric = df_metrics[df_metrics["metric"] == metric]
    
    # MTI primeiro
    df_mti = df_metric[df_metric["approach"] == "MTI"]
    fig.add_trace(
        go.Bar(
            name="MTI" if idx == 0 else None,
            x=df_mti["model"],
            y=df_mti["mean"],
            error_y=dict(type='data', array=df_mti["std"]),
            marker_color='rgb(55, 83, 109)',
            showlegend=(idx == 0),
            legendgroup="MTI"
        ),
        row=row, col=col
    )
    
    # STI depois
    df_sti = df_metric[df_metric["approach"] == "STI"]
    fig.add_trace(
        go.Bar(
            name="STI" if idx == 0 else None,
            x=df_sti["model"],
            y=df_sti["mean"],
            error_y=dict(type='data', array=df_sti["std"]),
            marker_color='rgb(26, 118, 255)',
            showlegend=(idx == 0),
            legendgroup="STI"
        ),
        row=row, col=col
    )
    
    # Configurar eixo Y
    fig.update_yaxes(range=[0, 5.5], row=row, col=col)

fig.update_layout(
    title_text="Comparação Detalhada: MTI vs STI por Métrica",
    height=900,
    barmode='group',
    template="plotly_white",
    legend=dict(x=0.7, y=0.05)
)

fig.show()

### 4.1 Heatmap - Médias por Métrica e Modelo

In [6]:
# Criar heatmaps separados para MTI e STI
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["MTI - Médias por Métrica", "STI - Médias por Métrica"],
    horizontal_spacing=0.15
)

# Heatmap MTI
df_mti_pivot = df_metrics[df_metrics["approach"] == "MTI"].pivot(
    index="model", columns="metric", values="mean"
)[METRICS]

fig.add_trace(
    go.Heatmap(
        z=df_mti_pivot.values,
        x=[METRIC_LABELS[m].replace('\n', ' ') for m in df_mti_pivot.columns],
        y=df_mti_pivot.index,
        colorscale='Blues',
        text=np.round(df_mti_pivot.values, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(x=0.45, len=0.9),
        zmin=1, zmax=5
    ),
    row=1, col=1
)

# Heatmap STI
df_sti_pivot = df_metrics[df_metrics["approach"] == "STI"].pivot(
    index="model", columns="metric", values="mean"
)[METRICS]

fig.add_trace(
    go.Heatmap(
        z=df_sti_pivot.values,
        x=[METRIC_LABELS[m].replace('\n', ' ') for m in df_sti_pivot.columns],
        y=df_sti_pivot.index,
        colorscale='Oranges',
        text=np.round(df_sti_pivot.values, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(x=1.02, len=0.9),
        zmin=1, zmax=5
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Heatmaps Comparativos: Médias das Métricas (MTI vs STI)",
    height=400,
    template="plotly_white"
)

fig.show()

## 5. Análise de Desvio Padrão

In [17]:
# Gráfico de desvio padrão para todas as métricas incluindo Overall
fig = go.Figure()

# Adicionar MTI primeiro
df_mti_all = df[df["approach"] == "MTI"]
for model in df_mti_all["model"].unique():
    df_model = df_mti_all[df_mti_all["model"] == model]
    
    # Ordenar métricas (individuais primeiro, Overall por último)
    metrics_order = METRICS + ["Overall (Média Geral)"]
    df_model = df_model.set_index("metric").loc[metrics_order].reset_index()
    
    fig.add_trace(go.Scatter(
        x=[METRIC_LABELS.get(m, m).replace('\n', ' ') for m in df_model["metric"]],
        y=df_model["std"],
        mode='lines+markers',
        name=f"{model} (MTI)",
        line=dict(width=2),
        marker=dict(size=8)
    ))

# Adicionar STI depois
df_sti_all = df[df["approach"] == "STI"]
for model in df_sti_all["model"].unique():
    df_model = df_sti_all[df_sti_all["model"] == model]
    
    # Ordenar métricas
    metrics_order = METRICS + ["Overall (Média Geral)"]
    df_model = df_model.set_index("metric").loc[metrics_order].reset_index()
    
    fig.add_trace(go.Scatter(
        x=[METRIC_LABELS.get(m, m).replace('\n', ' ') for m in df_model["metric"]],
        y=df_model["std"],
        mode='lines+markers',
        name=f"{model} (STI)",
        line=dict(width=2, dash='dash'),
        marker=dict(size=8)
    ))

fig.update_layout(
    title="Desvio Padrão das Métricas: MTI vs STI",
    xaxis_title="Métrica",
    yaxis_title="Desvio Padrão",
    height=500,
    template="plotly_white",
    legend=dict(x=1.02, y=1)
)

fig.show()

## 6. Contagem de Notas Máximas (Score = 5)

In [8]:
# Gráfico de barras empilhadas - Contagem de notas 5 por métrica
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[METRIC_LABELS[m] for m in METRICS] + ["Overall (Média Geral)"],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

row_col_map = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
metrics_to_plot = METRICS + ["Overall (Média Geral)"]

for idx, metric in enumerate(metrics_to_plot):
    row, col = row_col_map[idx]
    
    df_metric = df[df["metric"] == metric]
    
    # MTI primeiro
    df_mti = df_metric[df_metric["approach"] == "MTI"]
    fig.add_trace(
        go.Bar(
            name="MTI" if idx == 0 else None,
            x=df_mti["model"],
            y=df_mti["count_max_score"],
            marker_color='rgb(55, 83, 109)',
            showlegend=(idx == 0),
            legendgroup="MTI",
            text=df_mti["count_max_score"],
            textposition='outside'
        ),
        row=row, col=col
    )
    
    # STI depois
    df_sti = df_metric[df_metric["approach"] == "STI"]
    fig.add_trace(
        go.Bar(
            name="STI" if idx == 0 else None,
            x=df_sti["model"],
            y=df_sti["count_max_score"],
            marker_color='rgb(26, 118, 255)',
            showlegend=(idx == 0),
            legendgroup="STI",
            text=df_sti["count_max_score"],
            textposition='outside'
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Contagem de Notas Máximas (5) por Métrica: MTI vs STI",
    height=900,
    barmode='group',
    template="plotly_white",
    legend=dict(x=0.7, y=0.05)
)

fig.show()

## 7. Radar Chart - Perfil de Desempenho por Modelo

In [9]:
# Radar charts para cada modelo comparando MTI e STI
models = df["model"].unique()

for model in models:
    df_model = df_metrics[df_metrics["model"] == model]
    
    fig = go.Figure()
    
    # MTI primeiro
    df_mti = df_model[df_model["approach"] == "MTI"]
    df_mti = df_mti.set_index("metric").loc[METRICS]
    
    fig.add_trace(go.Scatterpolar(
        r=df_mti["mean"].tolist() + [df_mti["mean"].tolist()[0]],  # Fechar o polígono
        theta=[METRIC_LABELS[m].replace('\n', ' ') for m in METRICS] + [METRIC_LABELS[METRICS[0]].replace('\n', ' ')],
        fill='toself',
        name='MTI',
        line_color='rgb(55, 83, 109)',
        fillcolor='rgba(55, 83, 109, 0.3)'
    ))
    
    # STI depois
    df_sti = df_model[df_model["approach"] == "STI"]
    df_sti = df_sti.set_index("metric").loc[METRICS]
    
    fig.add_trace(go.Scatterpolar(
        r=df_sti["mean"].tolist() + [df_sti["mean"].tolist()[0]],  # Fechar o polígono
        theta=[METRIC_LABELS[m].replace('\n', ' ') for m in METRICS] + [METRIC_LABELS[METRICS[0]].replace('\n', ' ')],
        fill='toself',
        name='STI',
        line_color='rgb(26, 118, 255)',
        fillcolor='rgba(26, 118, 255, 0.3)'
    ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 5]
            )
        ),
        showlegend=True,
        title=f"Perfil de Desempenho: {model} (MTI vs STI)",
        height=500,
        template="plotly_white"
    )
    
    fig.show()

## 8. Análise de Diferenças (Δ = MTI - STI)

In [10]:
# Calcular diferenças para todas as métricas
diff_data = []

for model in df["model"].unique():
    for metric in METRICS + ["Overall (Média Geral)"]:
        df_model_metric = df[(df["model"] == model) & (df["metric"] == metric)]
        
        mti_mean = df_model_metric[df_model_metric["approach"] == "MTI"]["mean"].values[0]
        sti_mean = df_model_metric[df_model_metric["approach"] == "STI"]["mean"].values[0]
        
        diff_data.append({
            "model": model,
            "metric": metric,
            "difference": mti_mean - sti_mean,
            "mti_mean": mti_mean,
            "sti_mean": sti_mean
        })

df_diff = pd.DataFrame(diff_data)

# Gráfico de barras divergentes (positivo = MTI melhor, negativo = STI melhor)
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[METRIC_LABELS[m] for m in METRICS] + ["Overall (Média Geral)"],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

row_col_map = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
metrics_to_plot = METRICS + ["Overall (Média Geral)"]

for idx, metric in enumerate(metrics_to_plot):
    row, col = row_col_map[idx]
    
    df_metric_diff = df_diff[df_diff["metric"] == metric]
    
    # Cores baseadas em positivo (MTI melhor) ou negativo (STI melhor)
    colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in df_metric_diff["difference"]]
    
    fig.add_trace(
        go.Bar(
            x=df_metric_diff["model"],
            y=df_metric_diff["difference"],
            marker_color=colors,
            text=[f'{val:+.3f}' for val in df_metric_diff["difference"]],
            textposition='outside',
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Adicionar linha zero
    fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5, row=row, col=col)

fig.update_layout(
    title_text="Diferença de Médias (Δ = MTI - STI)<br><sub>Verde: MTI melhor | Vermelho: STI melhor</sub>",
    height=900,
    template="plotly_white"
)

fig.show()

In [11]:
# Heatmap de diferenças
df_diff_metrics = df_diff[df_diff["metric"] != "Overall (Média Geral)"]

df_diff_pivot = df_diff_metrics.pivot(
    index="model", columns="metric", values="difference"
)[METRICS]

fig = go.Figure(data=go.Heatmap(
    z=df_diff_pivot.values,
    x=[METRIC_LABELS[m].replace('\n', ' ') for m in df_diff_pivot.columns],
    y=df_diff_pivot.index,
    colorscale='RdYlGn',  # Vermelho (negativo/STI melhor) -> Verde (positivo/MTI melhor)
    text=np.round(df_diff_pivot.values, 3),
    texttemplate='%{text:+.3f}',
    textfont={"size": 11},
    zmid=0,  # Centro em zero
    colorbar=dict(title="Δ (MTI - STI)")
))

fig.update_layout(
    title="Heatmap de Diferenças: Δ = MTI - STI<br><sub>Verde: MTI superior | Vermelho: STI superior</sub>",
    xaxis_title="Métrica",
    yaxis_title="Modelo",
    height=400,
    template="plotly_white"
)

fig.show()

## 9. Tabelas Resumo - Dados Quantitativos

In [12]:
# Tabela resumo com todas as estatísticas
summary_rows = []

for model in df["model"].unique():
    for metric in METRICS + ["Overall (Média Geral)"]:
        df_model_metric = df[(df["model"] == model) & (df["metric"] == metric)]
        
        # MTI
        mti_data = df_model_metric[df_model_metric["approach"] == "MTI"].iloc[0]
        
        # STI
        sti_data = df_model_metric[df_model_metric["approach"] == "STI"].iloc[0]
        
        # Diferença
        diff = mti_data["mean"] - sti_data["mean"]
        
        summary_rows.append({
            "Modelo": model,
            "Métrica": metric,
            "MTI μ": f"{mti_data['mean']:.3f}",
            "MTI σ": f"{mti_data['std']:.3f}",
            "MTI #5": mti_data["count_max_score"],
            "STI μ": f"{sti_data['mean']:.3f}",
            "STI σ": f"{sti_data['std']:.3f}",
            "STI #5": sti_data["count_max_score"],
            "Δ (MTI-STI)": f"{diff:+.3f}",
            "Melhor": "MTI" if diff > 0 else "STI" if diff < 0 else "TIE"
        })

df_summary = pd.DataFrame(summary_rows)

print("📊 TABELA RESUMO COMPLETA - MTI vs STI")
print("="*120)
df_summary

📊 TABELA RESUMO COMPLETA - MTI vs STI


,Modelo,Métrica,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,coherence,4.629,0.504,1534,4.568,0.515,1385,+0.061,MTI
1,gpt-4o-mini,specificity,4.373,0.963,1274,4.332,0.949,1157,+0.041,MTI
2,gpt-4o-mini,informativeness,4.550,0.549,1384,4.514,0.537,1276,+0.036,MTI
3,gpt-4o-mini,relevance,4.959,0.203,2303,4.964,0.191,2315,-0.005,STI
4,gpt-4o-mini,Understandability,4.709,0.472,1720,4.644,0.496,1565,+0.065,MTI
5,gpt-4o-mini,Overall (Média Geral),4.644,0.622,8215,4.604,0.624,7698,+0.040,MTI
6,llama-3.3-70b-versatile,coherence,4.525,0.527,1293,4.414,0.579,1101,+0.110,MTI
7,llama-3.3-70b-versatile,specificity,4.277,0.942,1068,4.188,0.943,908,+0.090,MTI
8,llama-3.3-70b-versatile,informativeness,4.457,0.570,1189,4.368,0.591,1018,+0.089,MTI
9,llama-3.3-70b-versatile,relevance,4.956,0.210,2296,4.907,0.338,2204,+0.049,MTI


### 9.1 Tabelas Separadas por Métrica

In [13]:
# Imprimir tabelas separadas para cada métrica
for metric in METRICS + ["Overall (Média Geral)"]:
    df_metric_summary = df_summary[df_summary["Métrica"] == metric]
    
    print("\n" + "="*120)
    if metric == "Overall (Média Geral)":
        print(f"📊 {metric}")
    else:
        print(f"📊 {METRIC_LABELS.get(metric, metric).replace(chr(10), ' - ')}")
    print("="*120)
    display(df_metric_summary.drop(columns=["Métrica"]).reset_index(drop=True))


📊 Coherence - (Coerência)


,Modelo,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,4.629,0.504,1534,4.568,0.515,1385,+0.061,MTI
1,llama-3.3-70b-versatile,4.525,0.527,1293,4.414,0.579,1101,+0.110,MTI



📊 Specificity - (Especificidade)


,Modelo,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,4.373,0.963,1274,4.332,0.949,1157,+0.041,MTI
1,llama-3.3-70b-versatile,4.277,0.942,1068,4.188,0.943,908,+0.090,MTI



📊 Informativeness - (Informatividade)


...

,Modelo,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,4.959,0.203,2303,4.964,0.191,2315,-0.005,STI
1,llama-3.3-70b-versatile,4.956,0.210,2296,4.907,0.338,2204,+0.049,MTI



📊 Understandability - (Compreensibilidade)


,Modelo,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,4.709,0.472,1720,4.644,0.496,1565,+0.065,MTI
1,llama-3.3-70b-versatile,4.651,0.494,1583,4.549,0.549,1378,+0.102,MTI



📊 Overall (Média Geral)


,Modelo,MTI μ,MTI σ,MTI #5,STI μ,STI σ,STI #5,Δ (MTI-STI),Melhor
0,gpt-4o-mini,4.644,0.622,8215,4.604,0.624,7698,+0.040,MTI
1,llama-3.3-70b-versatile,4.573,0.638,7429,4.485,0.675,6609,+0.088,MTI


## 10. Análise de Dominância - Qual Abordagem Vence em Cada Métrica?

In [14]:
# Contar vitórias por métrica através de todos os modelos
dominance_data = []

for metric in METRICS + ["Overall (Média Geral)"]:
    df_metric = df_diff[df_diff["metric"] == metric]
    
    mti_wins = (df_metric["difference"] > 0).sum()
    sti_wins = (df_metric["difference"] < 0).sum()
    ties = (df_metric["difference"] == 0).sum()
    
    dominant = "MTI" if mti_wins > sti_wins else "STI" if sti_wins > mti_wins else "BALANCED"
    
    dominance_data.append({
        "Métrica": metric,
        "Vitórias MTI": mti_wins,
        "Vitórias STI": sti_wins,
        "Empates": ties,
        "Total Modelos": len(df_metric),
        "Abordagem Dominante": dominant
    })

df_dominance = pd.DataFrame(dominance_data)

print("🏆 ANÁLISE DE DOMINÂNCIA POR MÉTRICA")
print("="*100)
display(df_dominance)

# Gráfico de barras empilhadas
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Vitórias MTI',
    x=[METRIC_LABELS.get(m, m).replace('\n', ' ') for m in df_dominance["Métrica"]],
    y=df_dominance["Vitórias MTI"],
    marker_color='green',
    text=df_dominance["Vitórias MTI"],
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Vitórias STI',
    x=[METRIC_LABELS.get(m, m).replace('\n', ' ') for m in df_dominance["Métrica"]],
    y=df_dominance["Vitórias STI"],
    marker_color='red',
    text=df_dominance["Vitórias STI"],
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Empates',
    x=[METRIC_LABELS.get(m, m).replace('\n', ' ') for m in df_dominance["Métrica"]],
    y=df_dominance["Empates"],
    marker_color='gray',
    text=df_dominance["Empates"],
    textposition='inside'
))

fig.update_layout(
    barmode='stack',
    title="Distribuição de Vitórias por Métrica (considerando todos os modelos)",
    xaxis_title="Métrica",
    yaxis_title="Número de Modelos",
    height=500,
    template="plotly_white"
)

fig.show()

🏆 ANÁLISE DE DOMINÂNCIA POR MÉTRICA


,Métrica,Vitórias MTI,Vitórias STI,Empates,Total Modelos,Abordagem Dominante
0,coherence,2,0,0,2,MTI
1,specificity,2,0,0,2,MTI
2,informativeness,2,0,0,2,MTI
3,relevance,1,1,0,2,BALANCED
4,Understandability,2,0,0,2,MTI
5,Overall (Média Geral),2,0,0,2,MTI


## 11. Boxplots - Distribuição de Médias por Abordagem

In [15]:
# Boxplots para cada métrica mostrando a distribuição das médias entre modelos
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[METRIC_LABELS[m] for m in METRICS] + ["Overall (Média Geral)"],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

row_col_map = [(1,1), (1,2), (2,1), (2,2), (3,1), (3,2)]
metrics_to_plot = METRICS + ["Overall (Média Geral)"]

for idx, metric in enumerate(metrics_to_plot):
    row, col = row_col_map[idx]
    
    df_metric = df[df["metric"] == metric]
    
    # MTI
    fig.add_trace(
        go.Box(
            y=df_metric[df_metric["approach"] == "MTI"]["mean"],
            name="MTI" if idx == 0 else None,
            marker_color='rgb(55, 83, 109)',
            showlegend=(idx == 0),
            legendgroup="MTI",
            boxmean='sd'  # Mostrar média e desvio padrão
        ),
        row=row, col=col
    )
    
    # STI
    fig.add_trace(
        go.Box(
            y=df_metric[df_metric["approach"] == "STI"]["mean"],
            name="STI" if idx == 0 else None,
            marker_color='rgb(26, 118, 255)',
            showlegend=(idx == 0),
            legendgroup="STI",
            boxmean='sd'
        ),
        row=row, col=col
    )
    
    fig.update_yaxes(range=[0, 5.5], row=row, col=col)

fig.update_layout(
    title_text="Distribuição das Médias por Métrica (todos os modelos)",
    height=900,
    template="plotly_white",
    legend=dict(x=0.7, y=0.05)
)

fig.show()

## 12. Conclusões e Insights Finais

In [ ]:
# Gerar relatório final com insights
print("="*100)
print("📊 RELATÓRIO FINAL DE ANÁLISE COMPARATIVA: STI vs MTI")
print("="*100)

print("\n🎯 DESEMPENHO GERAL:")
print("-"*100)
for model in df["model"].unique():
    df_model_overall = df_overall[df_overall["model"] == model]
    
    mti_mean = df_model_overall[df_model_overall["approach"] == "MTI"]["mean"].values[0]
    sti_mean = df_model_overall[df_model_overall["approach"] == "STI"]["mean"].values[0]
    diff = mti_mean - sti_mean
    
    winner = "MTI" if diff > 0 else "STI" if diff < 0 else "EMPATE"
    print(f"\n{model}:")
    print(f"  • MTI: {mti_mean:.3f} | STI: {sti_mean:.3f} | Δ: {diff:+.3f}")
    print(f"  • Abordagem vencedora: {winner}")

print("\n\n🏆 MÉTRICAS MAIS IMPACTADAS:")
print("-"*100)

# Calcular diferenças médias por métrica
for metric in METRICS:
    df_metric_diff = df_diff[df_diff["metric"] == metric]
    avg_diff = df_metric_diff["difference"].mean()
    
    impact = "MTI superior" if avg_diff > 0.05 else "STI superior" if avg_diff < -0.05 else "Semelhantes"
    
    print(f"\n{METRIC_LABELS[metric].replace(chr(10), ' - ')}:")
    print(f"  • Diferença média: {avg_diff:+.3f}")
    print(f"  • Avaliação: {impact}")

print("\n\n📈 CONSISTÊNCIA (Desvio Padrão Médio):")
print("-"*100)

mti_avg_std = df[df["approach"] == "MTI"]["std"].mean()
sti_avg_std = df[df["approach"] == "STI"]["std"].mean()

print(f"MTI: {mti_avg_std:.3f}")
print(f"STI: {sti_avg_std:.3f}")
print(f"Abordagem mais consistente: {'MTI' if mti_avg_std < sti_avg_std else 'STI'}")

print("\n\n⭐ NOTAS MÁXIMAS (Score = 5):")
print("-"*100)

total_mti_5s = df[df["approach"] == "MTI"]["count_max_score"].sum()
total_sti_5s = df[df["approach"] == "STI"]["count_max_score"].sum()

print(f"Total MTI: {total_mti_5s}")
print(f"Total STI: {total_sti_5s}")
print(f"Diferença: {total_mti_5s - total_sti_5s:+d} (favorável a {'MTI' if total_mti_5s > total_sti_5s else 'STI'})")

print("\n" + "="*100)

📊 RELATÓRIO FINAL DE ANÁLISE COMPARATIVA: STI vs MTI

🎯 DESEMPENHO GERAL:
----------------------------------------------------------------------------------------------------

gpt-4o-mini:
  • MTI: 4.644 | STI: 4.604 | Δ: +0.040
  • Abordagem vencedora: MTI

llama-3.3-70b-versatile:
  • MTI: 4.573 | STI: 4.485 | Δ: +0.088
  • Abordagem vencedora: MTI


🏆 MÉTRICAS MAIS IMPACTADAS:
----------------------------------------------------------------------------------------------------

Coherence - (Coerência):
  • Diferença média: +0.086
  • Avaliação: MTI superior

Specificity - (Especificidade):
  • Diferença média: +0.065
  • Avaliação: MTI superior

Informativeness - (Informatividade):
  • Diferença média: +0.063
  • Avaliação: MTI superior

Relevance - (Relevância):
  • Diferença média: +0.022
  • Avaliação: Semelhantes

Understandability - (Compreensibilidade):
  • Diferença média: +0.083
  • Avaliação: MTI superior


📈 CONSISTÊNCIA (Desvio Padrão Médio):
------------------------------